# Test local chunking b4 moving to Setonix

In [ ]:
from pathlib import Path
import warnings
import dask
from dask.distributed import Client
import xarray as xr
import hdf5plugin

dpird_path= Path("C:/Users/John/OneDrive/Desktop/ICRAR/The_work/ICRAR-Weather-Forcasting/dataset_DPIRD_utc0_clean/DPIRD_final_stations_utc0.nc")
ecmwf_path= Path("C:/Users/John/OneDrive/Desktop/ICRAR/The_work/ICRAR-Weather-Forcasting/dataset_ecmwf_op_clean/2024/02/06.nc")

tmp_drop_path= Path("../.tmp/compressed")

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning
)

### DPIRD dataset

In [ ]:
dpird_ds=xr.open_dataset(dpird_path, engine="h5netcdf")
chunked_dpird_ds= dpird_ds.chunk({'station':96,'time':52624})
chunked_dpird_ds.attrs

### ECMWF dataset

In [ ]:
ecmwf_ds=xr.open_dataset(ecmwf_path, engine="h5netcdf")
chunked_ecmwf_ds= ecmwf_ds.chunk({"time": 4})
chunked_ecmwf_ds

## Chunking & Compression
Compression step is in encoding

In [ ]:
client= Client(n_workers=2, threads_per_worker=2)
client

In [ ]:
out_dir = tmp_drop_path
out_dir.mkdir(parents=True, exist_ok=True)

dpird_out = out_dir / "DPIRD_final_stations_utc0.chunked_zstd.nc"
ecmwf_out = out_dir / "2024_02_06.chunked_zstd.nc"

def build_var_encoding(ds, chunk_dict, complevel=5):
    enc = {}
    for v in ds.data_vars:
        var_chunks = tuple(chunk_dict.get(dim, ds[v].sizes[dim]) for dim in ds[v].dims)
        blosc_filter = hdf5plugin.Blosc(cname="zstd", clevel=complevel, shuffle=hdf5plugin.Blosc.SHUFFLE)

        enc[v] = {
            "compression": blosc_filter["compression"],
            "compression_opts": blosc_filter["compression_opts"],
            "chunksizes": var_chunks,
        }
    return enc

dpird_chunks = {'station':96,'time':52624}
ecmwf_chunks = {"time": 6}
                
dpird_encoding = build_var_encoding(dpird_ds, dpird_chunks)
ecmwf_encoding = build_var_encoding(ecmwf_ds, ecmwf_chunks)

# Dask-lazy netCDF writes
dpird_write = dpird_ds.to_netcdf(
    path=dpird_out,
    engine="h5netcdf",
    encoding=dpird_encoding,
    compute=False,
)

ecmwf_write = ecmwf_ds.to_netcdf(
    path=ecmwf_out,
    engine="h5netcdf",
    encoding=ecmwf_encoding,
    compute=False,
)

# Execute both writes in parallel
dask.compute(dpird_write, ecmwf_write)

| Dataset | Original Size | .to_netcdf() old chunks | .to_netcdf() new chunks
|---|---|---|---|
| DPIRD | 2.7GB | 563MB | 441MB |
| ECMWF | 4.0GB | 1.65GB | 1.71GB |

## Check encoding of tmp files

In [ ]:
tmp_chunked_dpird_path= Path("D:/Projects/S3-Kerchunk-streamer/.tmp/compressed/DPIRD_final_stations_utc0.chunked_zlib.nc")
tmp_chunked_ecmwf_path= Path("D:/Projects/S3-Kerchunk-streamer/.tmp/compressed/2024_02_06.chunked_zlib.nc")

In [ ]:
dask_chunked_dpird_ds= xr.open_dataset(tmp_chunked_dpird_path,engine="h5netcdf")
print(dask_chunked_dpird_ds.airTemperature.encoding)

In [ ]:
dask_chunked_ecmwf_ds= xr.open_dataset(tmp_chunked_ecmwf_path,engine="h5netcdf")
print(dask_chunked_ecmwf_ds.t250.encoding)